# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

In [1]:
# Install ChromaDB, Sentence-Transformers, and LangChain components
!pip install -q -U chromadb sentence-transformers langchain-community langchain-text-splitters

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# YOUR CODE HERE
import os
from google.colab import drive
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Mount Google Drive
drive.mount('/content/drive')

# Path to your mounted SOP folder in Google Drive
POLICIES_DIR = "/content/drive/MyDrive/sop_documents"

# 2. Initialize the HuggingFace Embedding Model
# Note: all-MiniLM-L6-v2 outputs 384-dimensional dense vectors
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"}  # Runs efficiently on CPU
)

# 3. Validate Embedding Model with a Sample String
sample_text = "Domestic orders deliver within 3-7 business days."
sample_embedding = embeddings.embed_query(sample_text)

print("\n--- Embedding Validation ---")
print(f"Sample Text         : '{sample_text}'")
print(f"Embedding Dimensions: {len(sample_embedding)} (Expected: 384)")
assert len(sample_embedding) == 384, f"Expected 384 dimensions, got {len(sample_embedding)}"
print("Validation Check    : PASSED!")

# 4. Load SOP Documents from Mounted Google Drive
if not os.path.exists(POLICIES_DIR):
    raise FileNotFoundError(
        f"Directory '{POLICIES_DIR}' not found. Please ensure your SOP folder exists at '{POLICIES_DIR}' in Google Drive."
    )

loader = DirectoryLoader(POLICIES_DIR, glob="**/*.md", loader_cls=TextLoader)
raw_documents = loader.load()
print(f"\nLoaded {len(raw_documents)} raw SOP document(s) from Drive.")

# 5. Chunk Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Split documents into {len(chunks)} text chunks.")

# 6. Embed Chunked Documents and Validate Output
chunk_texts = [doc.page_content for doc in chunks]
embedded_docs = embeddings.embed_documents(chunk_texts)

print("\n--- Document Vectorization Validation ---")
print(f"Total Chunks Embedded: {len(embedded_docs)}")
print(f"Vector Dimension     : {len(embedded_docs[0])}")
print("Status               : All SOP documents successfully vectorized into semantic space.")

/tmp/ipykernel_7234/706009208.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


/tmp/ipykernel_7234/706009208.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- Embedding Validation ---
Sample Text         : 'Domestic orders deliver within 3-7 business days.'
Embedding Dimensions: 384 (Expected: 384)
Validation Check    : PASSED!

Loaded 13 raw SOP document(s) from Drive.
Split documents into 64 text chunks.

--- Document Vectorization Validation ---
Total Chunks Embedded: 64
Vector Dimension     : 384
Status               : All SOP documents successfully vectorized into semantic space.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [6]:
import shutil
import os

from langchain_community.vectorstores import Chroma

# 1. Define Persistence Path and Initialize ChromaDB Index
PERSIST_DIR = "./chroma_db"
# Ensure absolute path for clarity and consistency
ABS_PERSIST_DIR = os.path.abspath(PERSIST_DIR)

print(f"Building persistent ChromaDB vector store at '{ABS_PERSIST_DIR}'...")

# Ensure a clean start by removing the previous ChromaDB directory if it exists
shutil.rmtree(ABS_PERSIST_DIR, ignore_errors=True)

# Explicitly create the directory to ensure write permissions
os.makedirs(ABS_PERSIST_DIR, exist_ok=True)

# Chroma.from_documents automatically computes embeddings and saves them to disk
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=ABS_PERSIST_DIR
)

print("ChromaDB vector index built and persisted successfully.")

# 2. Validate Index Count
# vector_db._collection.count() returns the total number of document chunks stored
total_vector_count = vector_db._collection.count()

print("\n--- Index Validation ---")
print(f"Total Chunks Ingested into ChromaDB: {total_vector_count}")
print(f"Expected Chunks: {len(chunks)}")
assert total_vector_count == len(chunks), f"Mismatch! Expected {len(chunks)} chunks, found {total_vector_count}."
print("Validation Check: PASSED!")

# 3. Test Retrieval with Similarity Search
test_query = "refund"
k_results = 1

print(f"\n--- Testing Similarity Search (query: '{test_query}', k={k_results}) ---")
retrieved_docs = vector_db.similarity_search(test_query, k=k_results)

for idx, doc in enumerate(retrieved_docs, 1):
    print(f"\n[Result {idx}]")
    print(f"Source Metadata: {doc.metadata}")
    print(f"Content Snippet:\n{doc.page_content[:300]}...")

Building persistent ChromaDB vector store at '/content/chroma_db'...
ChromaDB vector index built and persisted successfully.

--- Index Validation ---
Total Chunks Ingested into ChromaDB: 64
Expected Chunks: 64
Validation Check: PASSED!

--- Testing Similarity Search (query: 'refund', k=1) ---

[Result 1]
Source Metadata: {'source': '/content/drive/MyDrive/sop_documents/refund_policy.md'}
Content Snippet:
# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already cons...


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

In [ ]:
import shutil

# Copy outputs.json from Drive to Colab's working directory
shutil.copy("/content/drive/MyDrive/outputs.json", "./outputs.json")
print("Copied outputs.json from Google Drive!")

In [7]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Point directly to outputs.json stored in Google Drive
OUTPUTS_PATH = "/content/drive/MyDrive/outputs.json"

if not os.path.exists(OUTPUTS_PATH):
    raise FileNotFoundError(
        f"'{OUTPUTS_PATH}' not found in Google Drive. "
        f"Please make sure 'outputs.json' exists at MyDrive/outputs.json."
    )

with open(OUTPUTS_PATH, "r", encoding="utf-8") as f:
    outputs_data = json.load(f)

test_query = outputs_data.get("test_query")
print(f"Loaded test query from Google Drive: '{test_query}'")

# 2. Retrieve Top-1 Document (Naive RAG)
k_retrieval = 1
retrieved_docs = vector_db.similarity_search(test_query, k=k_retrieval)

top_doc = retrieved_docs[0]
retrieved_context = top_doc.page_content

print("\n--- Naive RAG Retrieval Output ---")
print(f"Retrieved Document Source: {top_doc.metadata.get('source', 'Unknown')}")
print(f"Retrieved Content Snippet:\n{retrieved_context}\n")

# 3. Load Qwen2.5-1.5B Model in 4-bit Precision
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Augment Prompt with Context
messages = [
    {
        "role": "system",
        "content": f"Answer strictly using this SOP: {retrieved_context}"
    },
    {
        "role": "user",
        "content": test_query
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 5. Generate Response
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

gen_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
naive_rag_output = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

print("--- Naive RAG LLM Response ---")
print(naive_rag_output)

# 6. Save updated dictionary directly back to Google Drive
outputs_data["naive_rag_output"] = naive_rag_output

with open(OUTPUTS_PATH, "w", encoding="utf-8") as f:
    json.dump(outputs_data, f, indent=4)

print(f"\nSuccessfully updated 'naive_rag_output' inside '{OUTPUTS_PATH}' on Google Drive.")

Loaded test query from Google Drive: 'My order is delayed. When will it arrive and can I get a refund for the delay?'

--- Naive RAG Retrieval Output ---
Retrieved Document Source: /content/drive/MyDrive/sop_documents/order_tracking.md
Retrieved Content Snippet:
## "Delivered But Not Received"
1. Confirm the shipping address on the order.
2. Ask the customer to check with neighbors, household members, and safe
   drop-off spots, and to allow 24 hours, as carriers sometimes scan early.
3. If still missing, open a carrier investigation and offer a replacement or
   refund per the shipping delays and refund procedures.



model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

--- Naive RAG LLM Response ---
If your order is delayed, here are some steps you can take:

1. **Check Shipping Address**: Ensure that the correct address was entered in your order confirmation.

2. **Contact Neighbors/Household Members**: Reach out to neighbors or family members who might have received the package. This could help confirm if the item has been delivered.

3. **Allow Carrier Investigation**: Sometimes, carriers may scan packages early due to busy schedules. Allow up to 24 hours for this process.

4. **Open Carrier Investigation**: If the package remains unclaimed after these checks, contact the carrier to initiate an investigation. They should provide details about why the delivery was late.

5. **Offer Replacement/Refund**: Based on the reasons provided by

Successfully updated 'naive_rag_output' inside '/content/drive/MyDrive/outputs.json' on Google Drive.


---
## Save Artifacts for Downstream Notebooks

In [8]:
# YOUR CODE HERE
import os

chroma_path = "./chroma_db"

print("ChromaDB exists:", os.path.exists(chroma_path))
print("Is it a directory:", os.path.isdir(chroma_path))

if os.path.exists(chroma_path):
    print("ChromaDB contents:")
    print(os.listdir(chroma_path))

ChromaDB exists: True
Is it a directory: True
ChromaDB contents:
['e7b48b00-68b5-4d9d-9a2c-624a6deee153', 'chroma.sqlite3']


In [9]:

print(os.path.abspath("./chroma_db"))

/content/chroma_db


In [10]:


source = "/content/chroma_db"
destination = "/content/drive/MyDrive/chroma_db"

# Remove destination if it already exists
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy existing ChromaDB to Google Drive
shutil.copytree(source, destination)

print("ChromaDB copied successfully to Google Drive!")
print("Location:", destination)
print("Contents:", os.listdir(destination))

ChromaDB copied successfully to Google Drive!
Location: /content/drive/MyDrive/chroma_db
Contents: ['e7b48b00-68b5-4d9d-9a2c-624a6deee153', 'chroma.sqlite3']


In [11]:
print("Exists:", os.path.exists(destination))
print("Is directory:", os.path.isdir(destination))

Exists: True
Is directory: True


In [12]:
from langchain_community.vectorstores import Chroma

PERSIST_DIR = "/content/drive/MyDrive/chroma_db"

vector_db_drive = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings
)

print("Vector count:", vector_db_drive._collection.count())

Vector count: 64


/tmp/ipykernel_7234/2866850574.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db_drive = Chroma(


In [13]:
results = vector_db_drive.similarity_search("refund", k=1)

print(results[0].page_content)

# Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier. Digital
goods and subscription fees already consumed are non-refundable except where
required by local consumer law.


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**